In [ ]:
# Cell 1: 匯入套件

# 載入函式庫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# 匯入資料集與模型建構所需模組
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Add
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation, Add,
    GlobalAveragePooling2D, Dense, Dropout
)

In [ ]:
# Cell 2: 載入與預處理 CIFAR-100 資料S
print("正在載入 CIFAR-100 資料集...")
(x_train, y_train), (x_test, y_test) = cifar100.load_data(label_mode='fine')

# 1. 影像正規化 (Normalization)
# 將像素值從 0~255 縮放到 0~1 之間
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 2. 標籤 One-hot Encoding
num_classes = 100
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print(f"訓練集影像: {x_train.shape}")
print(f"測試集影像: {x_test.shape}")

cifar100_labels = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
    'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
    'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
    'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
    'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
    'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose',
    'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake',
    'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
    'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout',
    'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm'
]

def show_sample_images(images, labels, num=10):
    rows = 2
    cols = num // rows
    plt.figure(figsize=(12, 6))
    indices = np.random.choice(range(len(images)), num)

    for i, idx in enumerate(indices):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(images[idx])

        label_id = labels[idx][0]
        label_name = cifar100_labels[label_id]

        plt.title(f"{label_name}", fontsize=12)
        plt.axis('off')

    plt.suptitle("CIFAR-100 Training Samples", fontsize=16)
    plt.tight_layout()
    plt.show()

# 呼叫函數顯示圖片
show_sample_images(x_train, y_train, num=10)

In [ ]:
# Cell 3: 定義模型結構
def build_deep_resnet():
    inputs = Input(shape=(32, 32, 3))
    
    x = Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2(5e-4))(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    def residual_block(x, filters, stride=1):
        shortcut = x
        if stride != 1 or shortcut.shape[-1] != filters:
            shortcut = Conv2D(filters, (1, 1), strides=stride, padding='same', kernel_regularizer=l2(5e-4))(shortcut)
            shortcut = BatchNormalization()(shortcut)
            
        fx = Conv2D(filters, (3, 3), strides=stride, padding='same', kernel_regularizer=l2(5e-4))(x)
        fx = BatchNormalization()(fx)
        fx = Activation('relu')(fx)
        
        fx = Conv2D(filters, (3, 3), padding='same', kernel_regularizer=l2(5e-4))(fx)
        fx = BatchNormalization()(fx)
        
        out = Add()([shortcut, fx])
        return Activation('relu')(out)

    # 深層架構：每個階段 4 個 Block
    x = residual_block(x, 128)
    x = residual_block(x, 128)
    x = residual_block(x, 128)
    x = residual_block(x, 128)
    
    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)
    x = residual_block(x, 256)
    x = residual_block(x, 256)
    
    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512)
    x = residual_block(x, 512)
    x = residual_block(x, 512)
    
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(100, activation='softmax', kernel_regularizer=l2(5e-4))(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_deep_resnet()
model.summary()

In [ ]:
# Cell 4: 編譯與訓練

batch_size = 128  
epochs = 250

learning_rate_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.1,  
    decay_steps=(50000 * 0.9 // batch_size) * epochs, 
    alpha=0.001 
)

optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9, nesterov=True)

model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train, y_train_cat,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_split=0.1, 
                    verbose=1)

In [ ]:
# Cell 5: 繪製訓練結果
# 評估測試集
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"測試集準確率 (Test Accuracy): {test_acc:.4f}")

# 繪圖
plt.figure(figsize=(12, 4))

# 繪製 Accuracy 曲線
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

# 繪製 Loss 曲線
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: 儲存模型
# 將訓練好的模型儲存為 keras 格式
model_name = "cifar100_resnet_midterm.keras"
model.save(model_name)
print(f"模型已成功儲存為: {model_name}")

# 重新載入模型：
# from tensorflow.keras.models import load_model
# loaded_model = load_model(model_name)

In [ ]:
# Cell 7: 全部預測 + 顯示準確度 + 存檔 + 隨機抽樣圖片
def evaluate_save_and_plot(model, x_test, y_test, class_names, num_images=5):
    # ===== 1. 預測全部測試資料 =====
    pred_probs = model.predict(x_test, verbose=0)
    pred_labels = np.argmax(pred_probs, axis=1)
    true_labels = np.argmax(y_test, axis=1)
    confidences = np.max(pred_probs, axis=1)

    # ===== 2. 計算整體準確度 =====
    accuracy = np.mean(pred_labels == true_labels)
    print(f"測試集準確度：{accuracy:.4f} ({accuracy:.2%})")

    # ===== 3. 存成 CSV =====
    df = pd.DataFrame({
        "id": np.arange(len(pred_labels)),
        "true_label": true_labels,
        "true_class_name": [class_names[label] for label in true_labels],
        "predict_label": pred_labels,
        "predict_class_name": [class_names[label] for label in pred_labels],
        "confidence": confidences
    })

    # ===== 儲存為csv檔 =====
    file_name = "mid_112370102_楊子力_cifar100_predictions.csv"
    df.to_csv(file_name, index=False, encoding="utf-8-sig")
    print(f"CSV 檔案已儲存：{file_name}")

    # ===== 4. 隨機抽樣圖片顯示 =====
    indices = np.random.choice(range(len(x_test)), num_images, replace=False)

    plt.figure(figsize=(15, 6))
    for i, idx in enumerate(indices):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(x_test[idx])

        true_label = true_labels[idx]
        pred_label = pred_labels[idx]
        confidence = confidences[idx]

        title_color = 'green' if pred_label == true_label else 'red'
        plt.title(
            f"True: {class_names[true_label]}\n"
            f"Pred: {class_names[pred_label]}\n"
            f"({confidence:.2%})",
            color=title_color
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

    return df, accuracy

# 執行
df_result, test_acc = evaluate_save_and_plot(
    model=model,
    x_test=x_test,
    y_test=y_test_cat,
    class_names=cifar100_labels,
    num_images=5
)

df_result.head()

In [ ]:
import tensorflow as tf
print("可用 GPU 列表:", tf.config.list_physical_devices('GPU'))
# 應該要看到 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]